In [47]:
!pip install -q -U transformers accelerate

In [48]:
import re
import torch
from transformers import pipeline

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


In [49]:
generator = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-1.7B-Instruct",
    device_map="auto",
    dtype="auto"
)

print("StudyMate AI model loaded successfully!")

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

StudyMate AI model loaded successfully!


In [50]:
def calculator(expression):
    try:
        result = eval(
            expression,
            {"__builtins__": {}},
            {}
        )

        return f"Calculator Result: {result}"

    except Exception:
        return "Sorry, I could not calculate that."

In [51]:
def concept_explainer(topic):

    prompt = f"""
You are StudyMate, a helpful AI tutor.

Explain this topic to a beginner:

Topic: {topic}

Use this structure:

1. Simple definition
2. Easy explanation
3. Real-world example
4. Small technical example

Keep the explanation clear and beginner-friendly.
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    response = generator(
        messages,
        max_new_tokens=250
    )

    generated = response[0]["generated_text"]

    if isinstance(generated, list):
        return generated[-1]["content"]

    return generated

In [52]:
def study_planner(subject):

    prompt = f"""
You are StudyMate, a helpful study planning assistant.

Create a simple 3-day study plan for:

Subject: {subject}

Use this structure:

Day 1:
- Learn the basic concepts

Day 2:
- Learn important concepts
- Practice

Day 3:
- Revise
- Practice questions
- Take a mini test

Keep the plan realistic and beginner-friendly.
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    response = generator(
        messages,
        max_new_tokens=250
    )

    generated = response[0]["generated_text"]

    if isinstance(generated, list):
        return generated[-1]["content"]

    return generated

In [53]:
def extract_calculation(text):

    text = text.lower()

    # Remove common words
    for word in [
        "calculate",
        "solve",
        "what is",
        "what's",
        "please",
        "compute"
    ]:
        text = text.replace(word, "")

    # Keep only mathematical characters
    expression = re.sub(
        r"[^0-9+\-*/().% ]",
        "",
        text
    )

    return expression.strip()

In [54]:
def choose_tool(user_input):

    text = user_input.lower()

    # Study planning
    if any(word in text for word in [
        "study plan",
        "study schedule",
        "study timetable",
        "learning plan",
        "revision plan",
        "study planner"
    ]):
        return "PLANNER"

    # Calculations
    if any(word in text for word in [
        "calculate",
        "solve",
        "compute",
        "add",
        "subtract",
        "multiply",
        "divide"
    ]):
        return "CALCULATOR"

    # Otherwise explain the topic
    return "EXPLAINER"

In [55]:
def studymate_agent(user_input):

    tool = choose_tool(user_input)

    # Calculator
    if tool == "CALCULATOR":

        expression = extract_calculation(user_input)

        if not expression:
            return "Please provide a mathematical expression."

        return calculator(expression)

    # Study planner
    elif tool == "PLANNER":

        # Try to extract the subject from the request
        subject = user_input

        for phrase in [
            "create a study plan for",
            "make a study plan for",
            "give me a study plan for",
            "create a study schedule for",
            "make a study schedule for",
            "study plan for"
        ]:
            if phrase in subject.lower():
                subject = subject.lower().split(
                    phrase,
                    1
                )[1].strip()
                break

        return study_planner(subject)

    # Concept explainer
    else:

        return concept_explainer(user_input)

In [56]:
print(calculator("25 + 75"))
print(calculator("15 * 12"))
print(calculator("100 / 4"))

Calculator Result: 100
Calculator Result: 180
Calculator Result: 25.0


In [57]:
result = studymate_agent(
    "Explain recursion in programming"
)

print(result)

[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1. Simple definition:
Recursion is a programming technique where a function calls itself repeatedly until it reaches a base case that stops the recursion. It's like a game of Russian dolls where each doll is a smaller version of the previous one.

2. Easy explanation:
Imagine you have a set of stairs. You can either climb one step at a time or jump two steps at a time. If you can only climb one step at a time, you would climb the entire staircase. But if you can jump two steps at a time, you can take a shortcut and avoid climbing the entire staircase.

In programming, a function can choose to either do something or call itself. If it can only do something, it will just do it. But if it can jump two steps at a time (or do something), it can call itself to avoid doing the thing.

3. Real-world example:
A recursive function can be used to calculate the factorial of a number. For example, the factorial of 5 (5! = 5 × 4 × 3 × 2 × 1) can be calculated by multiplying 5 by the factorial of 4 (

In [58]:
result = studymate_agent(
    "Create a study plan for Python"
)

print(result)

[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Day 1:
- Learn the basic concepts:
  - Understand what Python is and its history
  - Learn about the different types of variables in Python (integers, floats, strings, boolean, lists)
  - Understand the basic control structures in Python (if, else, elif, for, while)
  - Learn about functions and modules
  - Understand what a dictionary is and how to use it
  - Learn about the built-in functions in Python

Day 2:
- Learn important concepts:
  - Understand how to work with different data types (strings, lists, tuples, dictionaries)
  - Learn about loops (for loops, while loops)
  - Understand how to work with conditional statements (if/else, elif)
  - Learn about exception handling
  - Understand how to use functions and modules effectively

Day 3:
- Revise:
  - Go over the basics learned in the first two days and make sure you understand them well
  - Review the notes and textbook

Day 3:
- Practice questions:
  - Use online resources to solve Python problems
  - Try to solve problems o

In [59]:
print("=" * 50)
print("🤖 MINI STUDYMATE AI AGENT")
print("=" * 50)

print("""
I can help you with:

🧮 Mathematical calculations
📚 Technical concept explanations
📅 Study planning

Type 'exit' to stop.
""")

while True:

    user_input = input("You: ").strip()

    if user_input.lower() == "exit":
        print("\nStudyMate: Goodbye! 👋")
        break

    if not user_input:
        print("\nStudyMate: Please enter a question.")
        continue

    result = studymate_agent(user_input)

    print("\nStudyMate:")
    print(result)

    print("\n" + "-" * 50)

🤖 MINI STUDYMATE AI AGENT

I can help you with:

🧮 Mathematical calculations
📚 Technical concept explanations
📅 Study planning

Type 'exit' to stop.

You: exit

StudyMate: Goodbye! 👋


In [60]:
requirements = """transformers>=4.45.0
torch>=2.2.0
accelerate>=0.34.0
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt created successfully!")

requirements.txt created successfully!


In [61]:
%%writefile /content/Mini-StudyMate-AI-Agent/studymate_agent.py

import re
import torch
from transformers import pipeline


# --------------------------------------------------
# Load AI Model
# --------------------------------------------------

generator = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-1.7B-Instruct",
    device_map="auto",
    dtype="auto"
)


# --------------------------------------------------
# Calculator
# --------------------------------------------------

def calculator(expression):
    try:
        result = eval(
            expression,
            {"__builtins__": {}},
            {}
        )

        return f"Calculator Result: {result}"

    except Exception:
        return "Sorry, I could not calculate that."


# --------------------------------------------------
# Concept Explainer
# --------------------------------------------------

def concept_explainer(topic):

    prompt = f"""
You are StudyMate, a helpful AI tutor.

Explain this topic to a beginner:

Topic: {topic}

Use this structure:

1. Simple definition
2. Easy explanation
3. Real-world example
4. Small technical example

Keep the explanation clear and beginner-friendly.
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    response = generator(
        messages,
        max_new_tokens=250
    )

    generated = response[0]["generated_text"]

    if isinstance(generated, list):
        return generated[-1]["content"]

    return generated


# --------------------------------------------------
# Study Planner
# --------------------------------------------------

def study_planner(subject):

    prompt = f"""
You are StudyMate, a helpful study planning assistant.

Create a simple 3-day study plan for:

Subject: {subject}

Use this structure:

Day 1:
- Learn the basic concepts

Day 2:
- Learn important concepts
- Practice

Day 3:
- Revise
- Practice questions
- Take a mini test

Keep the plan realistic and beginner-friendly.
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    response = generator(
        messages,
        max_new_tokens=250
    )

    generated = response[0]["generated_text"]

    if isinstance(generated, list):
        return generated[-1]["content"]

    return generated


# --------------------------------------------------
# Extract Calculation
# --------------------------------------------------

def extract_calculation(text):

    text = text.lower()

    for word in [
        "calculate",
        "solve",
        "what is",
        "what's",
        "please",
        "compute"
    ]:
        text = text.replace(word, "")

    expression = re.sub(
        r"[^0-9+\-*/().% ]",
        "",
        text
    )

    return expression.strip()


# --------------------------------------------------
# Choose Tool
# --------------------------------------------------

def choose_tool(user_input):

    text = user_input.lower()

    if any(word in text for word in [
        "study plan",
        "study schedule",
        "study timetable",
        "learning plan",
        "revision plan",
        "study planner"
    ]):
        return "PLANNER"

    if any(word in text for word in [
        "calculate",
        "solve",
        "compute",
        "add",
        "subtract",
        "multiply",
        "divide"
    ]):
        return "CALCULATOR"

    return "EXPLAINER"


# --------------------------------------------------
# Main StudyMate Agent
# --------------------------------------------------

def studymate_agent(user_input):

    tool = choose_tool(user_input)

    if tool == "CALCULATOR":

        expression = extract_calculation(user_input)

        if not expression:
            return "Please provide a mathematical expression."

        return calculator(expression)

    elif tool == "PLANNER":

        subject = user_input

        for phrase in [
            "create a study plan for",
            "make a study plan for",
            "give me a study plan for",
            "create a study schedule for",
            "make a study schedule for",
            "study plan for"
        ]:

            if phrase in subject.lower():

                subject = subject.lower().split(
                    phrase,
                    1
                )[1].strip()

                break

        return study_planner(subject)

    else:

        return concept_explainer(user_input)


# --------------------------------------------------
# Run Application
# --------------------------------------------------

if __name__ == "__main__":

    print("=" * 50)
    print("🤖 MINI STUDYMATE AI AGENT")
    print("=" * 50)

    print("""
I can help you with:

🧮 Mathematical calculations
📚 Technical concept explanations
📅 Study planning

Type 'exit' to stop.
""")

    while True:

        user_input = input("You: ").strip()

        if user_input.lower() == "exit":

            print("\nStudyMate: Goodbye! 👋")
            break

        if not user_input:

            print("\nStudyMate: Please enter a question.")
            continue

        result = studymate_agent(user_input)

        print("\nStudyMate:")
        print(result)

        print("\n" + "-" * 50)

Writing /content/Mini-StudyMate-AI-Agent/studymate_agent.py
